# ADC Example 5 — Analog Watchdog (AWD)

**Target:** STM32F103C8T6 · ADC1 · Channel 0 (PA0)
**Focus:** Hardware threshold comparator — interrupt on out-of-window conversion results.

---
## Concept

The analog watchdog compares every conversion result against a **high threshold** (`HTR`) and a **low threshold** (`LTR`). If the result is outside the `[LTR, HTR]` window, the AWD flag is set and — if enabled — an interrupt fires. The CPU never needs to poll the ADC value[reference:8].

---
## CubeMX Configuration

### ADC1 Parameters
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled |
| `Continuous Conversion Mode` | **Enabled** (watchdog needs repeated conversions) |
| `Number Of Conversion` | 1 |
| Rank 1 | Channel 0, Sampling Time 55.5 Cycles |

### Watchdog Section
| Parameter | Setting |
| :--- | :--- |
| `Enable Analog WatchDog Mode` | **Checked** |
| `Watchdog Channel` | `ADC_CHANNEL_0` |
| `High Threshold` | e.g. **3000** (≈ 2.42 V) |
| `Low Threshold` | e.g. **500** (≈ 0.40 V) |

### NVIC
- Enable **ADC1 and ADC2 global interrupts**. Without this, the AWD interrupt never reaches the CPU.

> **CubeMX limitation:** The generated `MX_ADC1_Init()` does **not** automatically call `HAL_ADC_AnalogWDGConfig()`. You must add the watchdog configuration manually in user code, as shown below.

## User Code — Manual AWD Configuration

Place this **after** `MX_ADC1_Init()` and **before** starting the ADC.

In [ ]:
ADC_AnalogWDGConfTypeDef awd = {0};

awd.WatchdogMode   = ADC_ANALOGWATCHDOG_SINGLE_REG;  // monitor one channel
awd.Channel        = ADC_CHANNEL_0;
awd.ITMode         = ENABLE;                          // interrupt on violation
awd.HighThreshold  = 3000;                            // 12-bit: 0..4095
awd.LowThreshold   = 500;

if (HAL_ADC_AnalogWDGConfig(&hadc1, &awd) != HAL_OK) {
    Error_Handler();
}

## Start Conversion and Handle the Callback

The AWD callback has a **different name** from the normal conversion callback: `HAL_ADC_LevelOutOfWindowCallback()`.

In [ ]:
// Start continuous conversion with interrupt
HAL_ADC_Start_IT(&hadc1);

// AWD violation callback
void HAL_ADC_LevelOutOfWindowCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        // Over-voltage or under-voltage detected
        // e.g. set an alarm flag, toggle LED, shut down output
    }
}

## Threshold Values at 12-bit Resolution

| Threshold | Raw value | Voltage (3.3 V ref) |
| :--- | :--- | :--- |
| `LowThreshold` = 500 | 500 | 0.40 V |
| `HighThreshold` = 3000 | 3000 | 2.42 V |
| Mid-scale | 2048 | 1.65 V |

> At 12-bit resolution, valid thresholds are **0–4095** (`0x000`–`0xFFF`). At lower resolutions, the range shrinks accordingly[reference:9].

## Notes

- The AWD monitors **every conversion result** of the selected channel in hardware; latency is one conversion cycle.
- For multi-channel monitoring, use `ADC_ANALOGWATCHDOG_ALL_REG` and configure the watchdog to cover all regular channels.
- Changing thresholds on the fly is possible with `HAL_ADC_AnalogWDGConfig()` while conversions are ongoing, but changing the monitored channel or watchdog mode requires the ADC to be stopped first.